# HuggingFace Pipeline
### Pretraining + Fine-tuning with PEFT Comparison (LoRA, AdaLoRA, IA3)

In [1]:
# Install packages
import sys
!{sys.executable} -m pip install pydicom opencv-python-headless torch torchvision
!{sys.executable} -m pip install transformers peft accelerate scikit-learn pandas

In [2]:
# Imports
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

# HuggingFace
from transformers import (
    Trainer,
    TrainingArguments,
    EvalPrediction,
    PreTrainedModel,
    PretrainedConfig
)
from peft import (
    get_peft_model,
    LoraConfig,
    AdaLoraConfig,
    IA3Config,
    TaskType
)

from utils import (
    LIDCDataset,
    LIDCLabeledDataset,
    PatchEmbed,
    load_labels
)
from models.ctvit import CTViT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


## Pretraining with HuggingFace Accelerate

In [3]:
# Load unlabeled dataset
root_path = "Data/"   # change the path

dataset         = LIDCDataset(root_path)
pretrain_subset = torch.utils.data.Subset(dataset, range(10))
pretrain_loader = DataLoader(pretrain_subset, batch_size=2, shuffle=True)

print(f"Pretraining on {len(pretrain_subset)} samples")

Unlabeled dataset: 20 CT series
Pretraining on 10 samples


In [4]:
# Initialize model
patch_embed         = PatchEmbed(in_channels=3, embed_dim=1024, patch_size=16).to(device)
model               = CTViT(embed_dim=1024, depth=24, num_heads=16).to(device)
patch_pixels        = 16 * 16 * 3
reconstruction_head = nn.Linear(1024, patch_pixels).to(device)

In [5]:
# Mask tokens
def mask_tokens(tokens, mask_ratio=0.75):
    B, N, C = tokens.shape
    num_masked = int(N * mask_ratio)
    noise = torch.rand(B, N, device=tokens.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    mask_ids = ids_shuffle[:, :num_masked]
    mask = torch.zeros(B, N, dtype=torch.bool, device=tokens.device)
    mask.scatter_(1, mask_ids, True)
    masked_tokens = tokens.clone()
    masked_tokens[mask] = 0.0
    return masked_tokens, mask

In [6]:
# Pretraining with HuggingFace Accelerate
from accelerate import Accelerator

accelerator = Accelerator()

optimizer = torch.optim.Adam(
    list(model.parameters()) +
    list(patch_embed.parameters()) +
    list(reconstruction_head.parameters()),
    lr=1e-4
)

model, patch_embed, reconstruction_head, optimizer, pretrain_loader = accelerator.prepare(
    model, patch_embed, reconstruction_head, optimizer, pretrain_loader
)

num_epochs = 5
model.train()
reconstruction_head.train()

for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in pretrain_loader:
        tokens = patch_embed(batch)
        masked_tokens, mask = mask_tokens(tokens)

        output = model(
            masked_tokens,
            window_size=(7, 7),
            window_block_indexes=[],
            spatial_size=(14, 14),
            cls_embed=False,
        )

        pred   = reconstruction_head(output)
        target = reconstruction_head(tokens.detach())
        loss   = ((pred - target) ** 2)[mask].mean()

        optimizer.zero_grad()
        accelerator.backward(loss)
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}  Loss: {total_loss/len(pretrain_loader):.4f}")

print("\nPretraining done! Saving weights...")
torch.save({
    "model":       accelerator.unwrap_model(model).state_dict(),
    "patch_embed": accelerator.unwrap_model(patch_embed).state_dict(),
}, "pretrained_weights_hf.pth")
print("Saved → pretrained_weights_hf.pth")

Epoch 1/5  Loss: 0.1747
Epoch 2/5  Loss: 0.0207
Epoch 3/5  Loss: 0.0165
Epoch 4/5  Loss: 0.0113
Epoch 5/5  Loss: 0.0083

Pretraining done! Saving weights...
Saved → pretrained_weights_hf.pth


## Fine-tuning with PEFT

In [7]:
# CTViTConfig
class CTViTConfig(PretrainedConfig):
    model_type = "ctvit"

    def __init__(
        self,
        embed_dim=1024,
        depth=24,
        num_heads=16,
        num_classes=4,
        vocab_size=1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim   = embed_dim
        self.depth       = depth
        self.num_heads   = num_heads
        self.num_classes = num_classes
        self.vocab_size  = vocab_size

print("CTViTConfig defined")

CTViTConfig defined


In [8]:
# CTViTClassifier extending PreTrainedModel
class CTViTClassifier(PreTrainedModel):
    config_class = CTViTConfig

    def __init__(self, config: CTViTConfig):
        super().__init__(config)
        self.patch_embed = PatchEmbed(
            in_channels=3,
            embed_dim=config.embed_dim,
            patch_size=16
        )
        self.backbone = CTViT(
            embed_dim=config.embed_dim,
            depth=config.depth,
            num_heads=config.num_heads,
        )
        self.classifier = nn.Linear(config.embed_dim, config.num_classes)

    def forward(self, pixel_values=None, labels=None, **kwargs):
        from transformers.modeling_outputs import SequenceClassifierOutput

        tokens = self.patch_embed(pixel_values)
        output = self.backbone(
            tokens,
            window_size=(7, 7),
            window_block_indexes=[],
            spatial_size=(14, 14),
            cls_embed=False,
        )
        pooled = output.mean(dim=1)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

print("CTViTClassifier defined")

CTViTClassifier defined


In [9]:
# Load labels
df = load_labels("tcia-diagnosis-data-2012-04-20.xls")
print(f"Total labeled patients: {len(df)}")
print(f"Class distribution:\n{df['diagnosis'].value_counts().sort_index()}")

Total labeled patients: 157
Class distribution:
diagnosis
0    27
1    36
2    43
3    51
Name: count, dtype: int64


In [10]:
# Labeled dataset with stratified split
root_path = "/Volumes/Expansion/Data/manifest-1600709154662/LIDC-IDRI/"   # change the path

labeled_dataset = LIDCLabeledDataset(root_path, df)

indices = list(range(len(labeled_dataset)))
labels  = [labeled_dataset.samples[i][1] for i in indices]

train_idx, val_idx = train_test_split(
    indices, test_size=0.2,
    stratify=labels,
    random_state=42
)

train_dataset = torch.utils.data.Subset(labeled_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(labeled_dataset, val_idx)
print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")

Labeled dataset: 157 samples found
Train: 125  Val: 32


In [11]:
# HuggingFace Dataset wrapper
class HFDataset(TorchDataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        return {"pixel_values": image, "labels": label}

hf_train = HFDataset(train_dataset)
hf_val   = HFDataset(val_dataset)
print("HF datasets ready ✅")

HF datasets ready ✅


In [12]:
# Metrics
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"macro_f1": f1}

## All 3 PEFT Methods and Compare

In [ ]:
# Train and compare all 3 PEFT methods

results_table = {}

# build a fresh base model with pretrained weights
def build_base_model():
    config     = CTViTConfig(embed_dim=1024, depth=24, num_heads=16, num_classes=4)
    base_model = CTViTClassifier(config)
    ckpt = torch.load("pretrained_weights_hf.pth", map_location="cpu")
    base_model.backbone.load_state_dict(ckpt["model"])
    base_model.patch_embed.load_state_dict(ckpt["patch_embed"])
    return base_model

# train and evaluate a peft_model
def train_and_evaluate(peft_model, method_name):
    print(f"\n{'='*60}")
    print(f"  Training: {method_name}")
    print(f"{'='*60}")

    training_args = TrainingArguments(
        output_dir=f"./hf_output_{method_name.lower().replace(' ', '_')}",
        num_train_epochs=5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=1e-4,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_dir=f"./logs_{method_name.lower()}",
        logging_steps=10,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Evaluate
    predictions = trainer.predict(hf_val)
    preds  = predictions.predictions.argmax(axis=1)
    labels = predictions.label_ids
    f1     = f1_score(labels, preds, average="macro", zero_division=0)

    # Store result
    results_table[method_name] = {
        "f1":     f1,
        "preds":  preds,
        "labels": labels,
    }

    print(f"\n{method_name} — Macro F1: {f1:.4f}")
    return trainer


# LoRA
base_model  = build_base_model()
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    exclude_modules=["patch_embed"],
)
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()
train_and_evaluate(lora_model, "LoRA")


# AdaLoRA
base_model      = build_base_model()
steps_per_epoch = len(hf_train) // 4
total_steps     = 5 * steps_per_epoch   # 5 epochs (can change the value to 10)

adalora_config = AdaLoraConfig(
    init_r=16,
    target_r=8,
    lora_alpha=32,
    target_modules=["q", "k", "v"],
    lora_dropout=0.1,
    total_step=total_steps,
    tinit=10,
    tfinal=50,
    task_type=TaskType.SEQ_CLS,
    exclude_modules=["patch_embed"],
)
adalora_model = get_peft_model(base_model, adalora_config)
adalora_model.print_trainable_parameters()
train_and_evaluate(adalora_model, "AdaLoRA")


# IA3
base_model = build_base_model()
ia3_config = IA3Config(
    target_modules=["q", "k", "v", "fc1", "fc2"],
    feedforward_modules=["fc1", "fc2"],
    task_type=TaskType.SEQ_CLS,
    exclude_modules=["patch_embed"],
)
ia3_model = get_peft_model(base_model, ia3_config)
ia3_model.print_trainable_parameters()
train_and_evaluate(ia3_model, "IA3")


print("\nAll 3 methods trained successfully!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable params: 2,363,396 || all params: 305,466,376 || trainable%: 0.7737

  Training: LoRA


/Users/shreyaspatel/Downloads/GenAI/CT/menv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# Comparison

print("\n")
print("=" * 60)
print("         PEFT METHOD COMPARISON — MACRO F1 SCORES")
print("=" * 60)
print(f"{'Method':<15} {'Macro F1':>10} {'Result':>15}")
print("-" * 60)

best_method = max(results_table, key=lambda x: results_table[x]["f1"])

for method, result in results_table.items():
    marker = "  ← BEST" if method == best_method else ""
    print(f"{method:<15} {result['f1']:>10.4f} {marker}")

print("=" * 60)
print(f"\nBest method: {best_method} (F1 = {results_table[best_method]['f1']:.4f})")

print("\n")
print("=" * 60)
print("         DETAILED CLASSIFICATION REPORTS")
print("=" * 60)

for method, result in results_table.items():
    print(f"\n── {method} ──")
    print(classification_report(
        result["labels"],
        result["preds"],
        target_names=["Unknown", "Benign", "Malignant-Primary", "Malignant-Metastatic"],
        zero_division=0
    ))

In [ ]:
print("HuggingFace pipeline complete.")